## <span style="color: #eab308">Table of Contents</span>

- [What my_cartograph does and the resources it uses](#what-my-cartograph-does-and-the-resources-it-uses)
- [First Build: what it has and how to launch it](#first-build-what-it-has-and-how-to-launch-it)
- [Step-1: Landing page](#step-1-landing-page)

<a id="what-my-cartograph-does-and-the-resources-it-uses"></a>

## <span style="color: #eab308">What my_cartograph does and the resources it uses</span>

Status: the plan below is what the first build follows. **Definition changed after the first build: you can point at a zip _or_ at a folder.** That change is described here and **is not built yet**; the build in the last section still takes a zip only.

Reference code is the original Cartograph (`cartograph-main.zip`, phases 1–10) and its specs (`project-assets.zip`, phases 1–13); an existing Next.js project that already calls Groq is the reference for the AI call.

### What the app does

`my_cartograph` is a **local developer tool for one person on localhost**. You give it a TypeScript or JavaScript project as the **path to a zip or to a folder**, and it draws that project as an interactive **dependency map**: folders are boxes, imports are lines. It reads the code with a real parser; the AI never decides what is on the map.

**One rule everything rests on: every box and every line comes from really parsing the code.** The AI may explain what the parser found; it may not decide that two files are connected, and it may not walk the graph. A file that can't be parsed is skipped and counted; an import that can't be resolved is listed with a reason; a route is shown only if its method and full path are both written in the code. Absent beats approximate.

**Screens**

```
 LANDING PAGE  (/)                               MAP PAGE  (/projects/<id>)
┌───────────────────────────────────┐   ┌────────────────────────────────────────────────┐
│ cartograph        [system|light|dark]│   │ my-project     added 2026-10-19         [Re-run]│
├───────────────────────────────────┤   ├──────────┬─────────────────────────┬─────────────┤
│  Load a project                    │   │Categories│ Map │ Routes 12          │ summary or  │
│  [ path to a .zip or folder ] [Go] │   │· 214 files│                         │ selected    │
│  Paste a path, or drag a zip or    │   │Page routes│  folded boxes, open into │ file:       │
│  folder from Explorer into the box │   │Layouts …  │  panels with file rows   │ Structure | │
│  ▸ extract ▸ select ▸ parse ▸ store│   │Components │  green in, amber out     │ Explanation │
│                                    │   │Unclassif. │                         │ blast radius│
│  Stored projects                   │   │  11rem    │      fills the middle    │   20rem     │
│  my-project   214 files  today     │   │           │                         │             │
│  other-app     90 files  3d ago    │   │           │                         │             │
└───────────────────────────────────┘   └──────────┴─────────────────────────┴─────────────┘
```

| Capability | What it does |
|---|---|
| **Load a project** | One path at a time: a **.zip** (unpacked safely into a temporary folder) or a **folder** (read in place, never changed). Then the named stages (`extract → select → parse → store`; a folder has no extract stage) are shown live. A failure says why and in which stage. |
| **Stored projects list** | Below the load control, every project loaded before, with its name, size and when it was added. Click one to reopen it without loading again. Loading the same project twice opens the existing one. |
| **Parser** | ts-morph reads every `import`, `export … from`, `import("…")` with a literal, and `require("…")`. Each is resolved to a real file using the project's own `tsconfig` (aliases, index files, barrels). Every import ends as internal, external, excluded or unresolved, and every failure is listed with a reason. Fan-in and fan-out are counted per file. |
| **Framework adapters** | Detect Next.js, NestJS, Express, Docusaurus, React or none, and give files roles and "reached by" (so a page is never reported as unused). Routes for Next.js route handlers and NestJS controllers; Express and the rest show none rather than a guess. |
| **The map** | Folders fold into at most 24 boxes (the repository's shape decides the depth). Click a box to open it into a panel of file rows; click the header to fold it. Box height shows how many files depend on it. Layout is deterministic. |
| **Selection** | Select a file or box: it and its edges stay bright, the rest dims. Green is flowing in, amber is flowing out. Hover works between map and pane. |
| **Detail pane** | With nothing selected: a summary of the repository (framework, counts, most depended-on files, files nothing imports, how many files no convention identified). With a file selected: path, kind, role, length, what it imports, what imports it, exports, and every path clickable. |
| **Blast radius / Dependency chain** | Two buttons: what breaks if this file changes, and what this file needs. Two levels by default, instant, no network. |
| **Insights** | Four facts from the import graph in a collapsed panel: files nothing imports, files imported by unusually many, import cycles, files over 1,000 lines. Fixed sentences, no model, no scores. |
| **Category rail** | Roles for the detected framework in a fixed reading order, with counts. Click one and everything else dims. |
| **Route table** | Method, full pattern and declaring file, plus the routes it saw but could not read exactly, with reasons. |
| **Explain** | For a file or a folded folder, the AI writes a short paragraph from the file's source and its real neighbours. Every path in the answer becomes a link that moves the map. Answers are cached, so the second click is instant and is marked as cached. |
| **Label unknown files** | For a file no convention identified, the AI picks one of seven non-structural roles (service, repository, model, util, config, component, hook) or says none. It can never say page, route or controller. |
| **Theme** | System, light or dark, remembered. Dense developer look: small type, monospace paths, colour only for direction and kind, nothing moves unless clicked. |

**What it deliberately does not do:** no sign-in or accounts, no scores, grades or "issues found", no private-repository fetching or GitHub downloads, no languages other than TypeScript and JavaScript, no AI deciding connections.

**Later, once the above works (specs exist, no original code):** evals (a check that an explanation names no file that does not exist), and an "ask the repository" agent. Both are outside the first build.

### Change of definition: point at a zip or at a folder

**What I understand:** the one thing you give the app is a **path**. It can name a `.zip`, or it can name a **folder** such as `C:\Users\rush_\OneDrive\APP_Cursor_Clone`. Everything after that (parsing, the map, Explain, the stored-projects list) is the same.

**Why a path box, not a file chooser.** A browser never reveals where a chosen file lives, so a chooser can only *upload* it. That is what the first build does for zips, and it is fine for a 5 MB zip but wasteful for a 483 MB one. A folder is worse: the browser's "choose a folder" uploads every file in it, `node_modules` included. This app runs on the same machine as its server, so the server can simply read the path. That avoids uploading anything, and it means a zip is also read where it sits.

| | A **zip** path | A **folder** path |
|---|---|---|
| What happens first | Unpacked into a temporary folder (code and JSON for real, empty stand-ins for images and the like) | Nothing. The parser walks the folder where it is |
| Stages shown | Unpack zip → Select → Parse → Store | Select → Parse → Store |
| Safety rules | Refuse absolute paths, `..`, symlinks, entries whose size lies; limits on entries and bytes | Symlinked folders are not followed (the walker lists them as excluded); the path must exist and be a folder |
| Your files | The zip is **never changed or deleted** (the app reads it in place) | **Read-only**: nothing in the folder is written or deleted |
| `node_modules`, `.git`, `.next`, other dot-folders | Not unpacked, listed as excluded | Not walked, listed as excluded |
| After storing | The temporary unpacked copy is deleted | Nothing to clean up |
| Project name | The zip's file name without `.zip` | The folder's name |

**What a folder of several projects does.** The parser already treats every folder with a framework-bearing `package.json` as its own *project* with its own adapter. I ran it on a folder holding one Next.js project (plus two zips beside it) from the terminal (no build change needed for that): it found the project as a nested **Next.js** project and gave the same numbers as the project's own zip: **140 code files found, 136 parsed, 4 skipped, 205 edges, 0 unresolved, 10 routes**, taking about 11 seconds (the folder is in OneDrive). The two zips beside it are **not opened**: a zip found inside a folder is just a non-code file, so it is ignored.

**Consequences worth knowing**
- **Paths on the map carry the folder.** Pointing at a folder that *contains* the project makes every path start with the project's folder name; pointing at the project's own folder (or at its zip, whose wrapper folder is stripped) gives paths from the project root.
- **Same code, same project: only when the paths match.** The project id is a hash of the file paths and contents. A project's zip and the project's own folder therefore give **one** id; the folder that contains it gives **another** (different prefix). That is correct, but it will look like a duplicate in the list.
- **A folder changes; a zip doesn't.** Loading the same folder after editing it gives a new id, so the old stored result stays beside the new one. Replacing it instead is a possible later feature.
- **A typed path is trusted.** The server reads whatever path you give it. On `localhost`, for one person, that is the point; it would not be acceptable on a shared server.

**What would change in the code** (not done): replace the upload on the landing page with one path field; the route handler takes `{ path }` instead of a request body, checks it is a file ending in `.zip` or a folder, and runs the same four stages with the unpack stage skipped for a folder; the upload code and its size limit go; `scripts/load.ts` already takes a path and would accept a folder too.

**Decided: a path box only, no "Browse…" button.** One path box is one obvious way. The landing page says what to do in a line under the box: **"Paste a path, or drag a zip or folder from Explorer into the box."** (Whether a drag from Explorer fills in the real path depends on the browser; that is checked when it is built. Pasting always works.)

### How a project flows through it

![How a project flows through my_cartograph: a path to a zip or a folder goes through the landing page and route handler, a zip is unpacked and a folder is read in place, then select, parse and store write data/project-id, and the map page and Explain read from there](SVG%20How%20a%20project%20flows%20through%20it/project-flow.svg)

The diagram has three parts: **A** is how a path becomes a stored project, **B** is what is written, and **C** is what reads it later.

**A · From a path to a stored project**

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **The landing page takes one path.** A `.zip` or a folder. Nothing is uploaded; the page sends the path to the server.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **The route handler checks it and picks a branch.** Does the path exist, is it a zip or a folder? If it is neither, the page says so at once and nothing starts.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **Zip branch: Stage 1, unpack.** Each entry is checked before it is written (no absolute paths, no `..`, no symlinks, no entry that inflates past its declared size). `node_modules` and dot-folders are not unpacked but are created empty so the parser still reports them as excluded. Images, stylesheets and the like become empty stand-ins, so an import of one still resolves to something that exists.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **Folder branch: nothing to unpack.** The parser walks the folder in place and never writes to it.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **Stages 2 to 4 are the same for both.** *Select* decides which files become nodes (skipped files are counted with a reason). *Parse* reads every import with ts-morph and resolves each to a real file or reports it unresolved. *Store* writes the project folder.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **Each stage line is streamed back** to the page as it happens (the dashed line on the left of the diagram). Nothing polls.

**B · What is written**

`data/<project id>/` gets `meta.json` (what the list shows), `result.json` (the parser's output) and `sources.json` (the text of the parsed files, so Explain still works after the source is gone). The id is a hash of the project's files. The temporary unpacked copy of a zip is deleted; the zip itself and the folder are never touched.

**C · What reads it later**

The **map page** reads `result.json` and checks it against the parser's contract before drawing; every graph calculation after that happens in the browser with no requests. **Explain** runs on the server on click: it reads the file's text from `sources.json`, checks the hash, looks in `ai-cache.json`, and only on a miss calls Groq.

> **The one idea to take away:**
> ```
> path ─► [unpack if zip] ─► select ─► parse ─► store ─► data/<id>/
>                                                            │
>                              map page (browser maths)  ◄───┤
>                              Explain (server, cached)  ◄───┘
> ```
> Where the code came from only matters up to the store stage. After that, a project is three files.

### Resources it will use

**Stack**

| Layer | Resource | Used for |
|---|---|---|
| Framework | Next.js 16 (App Router), React 19, TypeScript (strict) | Pages, server routes, UI |
| Styling | Tailwind CSS v4 | The dense developer look, light and dark tokens |
| Parser | `ts-morph` | Reading imports and exports from real syntax trees |
| Map | `@xyflow/react` (React Flow) and `@dagrejs/dagre` | Drawing the graph and laying it out |
| AI | `ai` (Vercel AI SDK), `@ai-sdk/groq`, `zod` | One call path for explain-file, explain-folder and classify-file; `zod` for the role answer |
| Zip | **No package**: a small reader on Node's own zlib (`lib/zip.ts`) | Unpacking a zip safely |
| Package manager | pnpm | As in the original |

All of these are **not installed**. I will name each package and what it is for, and wait, before installing anything.

**AI**
- Provider: **Groq**, model **`openai/gpt-oss-20b`**, for all three tasks (the same model an existing Next.js project of yours already uses).
- Key: **`GROQ_API_KEY`** in `.env.local`. I will tell you exactly what and where when we reach the first AI call. I won't copy it and it is never written in the notebook.
- Used only by Explain and role labelling. Everything else works with no key and no network.
- Cache: keyed on a hash of the question, the model name and a prompt version, stored in the project's `ai-cache.json`. **LangSmith is dropped**, so there is no tracing; a cached answer is marked as cached instead.

**Storage**
- Plain files in a git-ignored `data/` folder, one folder per project: `meta.json`, `result.json` (the parser's output, validated every time it is read), `sources.json` (the text of the parsed files) and `ai-cache.json` (made on first Explain). **No database.**
- **A zip is never stored.** It is read, and once the project is stored nothing of it remains in `data/`. A folder is read in place and never copied.
- `sources.json` holds only files the parser read, so `data/` stays small (about 470 KB for a 136-file project), which matters because the project sits in OneDrive. When Explain needs a file, its text comes from here and its hash must match the one the parser stored.

**External services:** Groq (Explain only). Nothing else: no GitHub, no auth provider, no hosted database.

**Reference material**
- `cartograph-main.zip`: the original code, phases 1–10.
- `project-assets.zip`: the video kit, with `CLAUDE.md`, the project doc and the 13 phase specs.
- An existing Next.js project of yours that already calls Groq: how the AI call is made (`generateText`, `Output.object`, Groq, `reasoningEffort: "low"`).
- The video "Nobody Can Read AI Code Anymore" (Cartograph, JavaScript Mastery).

**Removed from the original:** Clerk, Supabase (Postgres, row-level security, Realtime), `tar` / GitHub fetching, OpenAI, LangSmith, `proxy.ts`, organizations.

**Still to settle**
1. **Settled:** no zip package; the reader is written on Node's built-ins.
2. **Settled:** "the same project" is the hash of the files' paths and contents, whatever the zip was called.
3. **Done:** `CLAUDE.md` was rewritten for this plan.
4. **Settled:** a path box only, no file chooser ("Browse…"). The landing page says: paste a path, or drag a zip or folder from Explorer into the box.

<a id="first-build-what-it-has-and-how-to-launch-it"></a>

## <span style="color: #eab308">First Build: what it has and how to launch it</span>

The first build of `my_cartograph` exists and runs. It follows the plan above, with the changes listed under *What differs from the plan*. The code is in `my_cartograph/`, next to this notebook's folder.

### Is this a Next.js app? Yes

`my_cartograph` is a **Next.js 16.3.6 app** (App Router, Turbopack) with React 19 and TypeScript. `npm run dev` is just `next dev`, and `npm run build` is `next build`. It is not a Vite app: Next.js is the server, the bundler and the router, and the file system is the router.

| Next.js piece | Where | What it does here |
|---|---|---|
| **Layout** | `app/layout.tsx` | The shell around every page: `<html>`, fonts, and the theme read from a cookie so the first HTML already has the right theme. |
| **Pages** (server components) | `app/page.tsx`, `app/projects/[id]/page.tsx` | The landing page and the map page. They run on the server, so they can read `data/` directly. `[id]` is a dynamic route: the folder name becomes the project id. |
| **Route handler** | `app/api/projects/route.ts` | Receives the zip as the request body and streams the stage updates back, one JSON line per stage. |
| **Server actions** | `app/projects/[id]/actions.ts` | The Explain buttons call these straight from the browser; they run on the server, where the Groq key and the files are. |
| **Client components** (`"use client"`) | `components/load-form.tsx`, `analysis-view.tsx`, the map, the pane | Everything interactive. The graph work (folding, highlighting, blast radius) runs here, in the browser, with no requests. |

There is no Convex, Clerk or Inngest here: one Next.js process and plain files in `data/`. The Next.js version here is newer than most documentation, so the rule in `AGENTS.md` applies: read the guide in `node_modules/next/dist/docs/` before relying on memory.

### How to launch it

**First time on a machine** (the packages are already installed on this one):

```powershell
cd C:\Users\rush_\OneDrive\APP_Cartograph\my_cartograph
npm install
```

**Every time:**

```powershell
cd C:\Users\rush_\OneDrive\APP_Cartograph\my_cartograph
npm run dev
```

Then open **http://localhost:3000**. To stop it, press `Ctrl+C` in that terminal. If something else already uses port 3000, Next.js says so and picks the next free one; use the address it prints.

**What you need:**
- Node 24 (this machine has 24.20). No database, no account, no Docker.
- `.env.local` in `my_cartograph/` with one line, `GROQ_API_KEY=...`. It was copied from that other project's `.env`. Without it everything works **except Explain** and role labelling, which say so in words. The file is git-ignored; the key is not in this notebook.

### Using it

1. On the landing page choose a zip and click **Analyse**. The steps tick past: **Upload** (percent) → **Unpack zip** → **Select files** → **Parse imports** → **Store project**. A failure names the step and says why.
2. You land on the map. The **stored projects** list on the landing page now includes it; click any entry to reopen it without loading the zip again.
3. Loading the same code twice (even from a differently named zip) opens the stored copy instead of making a second one.

**On the map:**

| Where | What to try |
|---|---|
| **Map** | Click a folded box to open it into a panel of file rows; click the panel header to fold it. Scroll the wheel inside a panel with more than 14 files. Click a file to select it: its edges light up, **green flows into it, amber flows out of it**, everything else dims. Click empty canvas to deselect. |
| **Right pane, nothing selected** | The repository summary: framework, counts, most depended-on files, files nothing imports, files no convention identified, and a collapsed **Insights** panel. |
| **Right pane, a file selected** | **Structure** tab: kind, role, folder, length, depends on, depended on by, exports, both full lists, and the **Blast radius** and **Dependency chain** buttons. **Explanation** tab: the **Explain** button. Every path is clickable and moves the map. |
| **Left rail** | Roles for the detected framework with counts. Click one to dim everything that is not in it. |
| **Routes tab** (above the map) | Method, full pattern and declaring file, plus routes it saw but could not read exactly. |
| **Theme** (top right) | system / light / dark, remembered. |

### Terminal tools

These run the same code with no web app:

```powershell
npm run load -- "C:\path\to\project.zip"      # load a zip into data/ the way the landing page does
npm run parse -- <folder>                       # parse any folder and print files, coverage, routes
npm run parse -- --read data\<id>\result.json   # re-check a stored result against the contract and print its summary
npm run typecheck                               # tsc --noEmit
npm run lint
npm run build
```

### What is stored, and where

One folder per project in `my_cartograph/data/` (git-ignored):

```
data/
  <project id>/            ← 16 hex characters, a hash of the project's files
    meta.json              ← name, when added, counts: what the list shows
    result.json            ← the parser's output, checked against its contract on every read
    sources.json           ← the text of every parsed file, so Explain can read it
    ai-cache.json          ← explanations and role labels, made on first Explain click
    model-roles.json       ← roles the model gave unclassified files
```

**The zip is deleted as soon as the run ends**, whatever the outcome; your original zip is never touched. While running, the code is unpacked into the system temp folder (not into `data/`) and removed afterwards. The folder is small: a 483 MB zip of a Next.js project becomes a stored project of about 470 KB.

### What is in the code

About 6,600 lines of TypeScript. The parser, graph maths, map, detail pane and prompts are the original's design, carried over; the zip reader, the store, the ingest run and the landing page are new.

| Folder | What it is |
|---|---|
| `lib/parser/` | The standalone parser (ts-morph) and the six framework adapters. Imports no Next, React or file store. |
| `lib/graph/` | Pure graph functions: folding, view, highlight, blast radius, insights, labels. |
| `lib/zip.ts` | **New.** A zip reader on Node's own zlib (central directory, stored and deflated entries, zip64). Refuses encrypted entries, other compression, and entries that inflate larger than their header says. |
| `lib/ingest/` | **New.** `run.ts` is the four stages; `stages.ts` is the stage list the browser also reads. Refuses entries with absolute paths, `..` or drive letters; skips symlinks; skips `node_modules` and dot-folders (creating them empty so the parser still lists them as excluded); writes empty stand-ins for images and the like so imports of them still resolve. Limits: 500,000 entries, 1 GB of code and JSON, 4 GB upload. |
| `lib/store/projects.ts` | **New.** Everything under `data/`: list, read, write, cache, model roles, sources. |
| `lib/ai/` | `client.ts` is the one place a model is chosen (Groq, `openai/gpt-oss-20b`); `tasks.ts` has explain-file, explain-folder and classify-file behind the cache; `prompts.ts` is the original's. |
| `lib/analysis/context.ts` | What the model is handed, rebuilt from the stored result. The browser only names a file or folder. |
| `app/` | `/` landing page, `/projects/[id]` map page, `api/projects` (receives the zip, streams the stages back), and the Explain server actions. |
| `components/` | The map (`map/`), detail pane, rail, route table, explanation panel, load form, theme control. |
| `scripts/` | The terminal tools above. |

### What was checked

From the terminal, on a 483 MB zip of a Next.js project (the test project):
- Types, lint and the production build pass.
- Loading takes about a second. 71,690 zip entries; **140 code files found, 136 parsed, 4 skipped** (all `.d.ts` files, each with its reason); **496 imports seen: 205 internal, 279 external, 12 excluded, 0 unresolved**; 205 edges.
- Detected **Next.js**; 21 files reached without an import (7 pages, 8 route handlers, 2 layouts, 1 proxy, 3 config files); **10 routes** (all in `src/app/api`).
- Explain against Groq: the first call took about 0.75 s, the second came from the cache in 2 ms. Role labelling of an unclassified file worked.
- All 136 stored sources hash to the value the parser recorded. Loading the same zip through the web endpoint leaves nothing but the project folder in `data/`.

**Not checked: anything that needs a browser** (how the map looks, clicking, scrolling, the pane). That is yours to run.

### What differs from the plan above

| Plan said | Built |
|---|---|
| Keep the zip in the project folder; Explain reads single files out of it | **Zip is deleted after loading**; `sources.json` holds the text of the parsed files instead. |
| A "stale" marker for runs that die mid-way | Not built: nothing is listed until a run finishes, so there is nothing stale to show. |
| Re-run button on a project | Not built: reload the zip. |
| "Repository moved on" freshness note | Not built: a zip has no commit. |
| Package manager pnpm | **npm** (pnpm isn't installed here). |
| Point at a zip **or a folder**, by path | **Not built yet.** The first build takes a zip uploaded from a file chooser. The change is described in *Change of definition* above. |
| A zip-reading package to be chosen | **None**: written on Node's built-ins. |
| Coverage banner with a percentage | Not built, as in the original's code. The summary shows "N skipped" and "N unresolved". |

### Not in this build

Evals, the "ask the repository" agent, an option to delete a stored project from the list (delete its folder in `data/` by hand), and anything for private repositories or other languages. `LangSmith` tracing was dropped as decided.

<a id="step-1-landing-page"></a>

## <span style="color: #eab308">Step-1: Landing page</span>

The landing page is the first thing built and the only page you see before a project exists. It does two jobs: **load a project**, and **list every project loaded before**. Everything else (the map, Explain) starts from a click here.

> **How to read this section.** It is written from the code as committed (`41e683d`, "First build of my_cartograph"), not from a record of the order the files were typed. The parts below are in the order you would build them if you started from an empty Next.js app: shell → header → list → form → wiring. The landing page here still takes a **zip from a file chooser**; the path-box change in *Change of definition* is not built yet, and the last cell says what it will touch.

### What the landing page has

```
┌────────────────────────────────────────────────┐
│ cartograph                [system|light|dark]  │ ← header (app-header.tsx)
├────────────────────────────────────────────────┤
│  Load a project                                │
│  [ Choose File  no file chosen ]  [Analyse]    │ ← load-form.tsx
│  A zip of a TypeScript or JavaScript project…  │
│  ● Upload          42%                         │ ← appears after the click
│  ○ Unpack zip                                  │
│  ○ Select files                                │
│  ○ Parse imports                               │
│  ○ Store project                               │
│                                                │
│  Stored projects                            2  │ ← page.tsx
│  Project         Files Imports Routes   Added  │
│  my-project        136     205     10   3h ago │
│  other-app          90     120      0   3d ago │
└────────────────────────────────────────────────┘
```

| Feature | How it works | File |
|---|---|---|
| **Shell** | `<html>` and `<body>` for every page; fonts; the theme written into `<html>` before the browser paints. | `app/layout.tsx` |
| **Look** | Colour tokens defined once, with a light and a dark value each; every size in `rem`. | `app/globals.css` |
| **Header** | The name `cartograph` (a link back to `/`) and the theme control. | `components/app-header.tsx`, `theme-control.tsx` |
| **Load a project** | Pick a zip, click **Analyse**. The zip goes to the server as the request body. | `components/load-form.tsx` |
| **Live stages** | A five-line list (Upload + four stages) that ticks as the server reports each one. A failure names the stage and says why. | `load-form.tsx`, `state-mark.tsx`, `lib/ingest/stages.ts` |
| **Receiving end** | Saves the zip, runs the four stages, streams one JSON line back per stage change. | `app/api/projects/route.ts` |
| **Stored projects** | A table of every project in `data/`, newest first, each name a link to its map. Shows "N skipped, N unresolved" beside a name only when there is something to say. | `app/page.tsx`, `lib/store/projects.ts`, `lib/time.ts` |

**Server or browser?** `layout.tsx`, `page.tsx` and `app-header.tsx` are **server components**: they run on the server and can read `data/` and cookies directly. Only `load-form.tsx` and `theme-control.tsx` start with `"use client"`, because they need state and click handlers. Keeping the client parts small means the list is plain HTML by the time it arrives.

### What the files look like

```
app/
├── layout.tsx            ← shell: fonts, <html data-theme>
├── globals.css           ← colour tokens, rem scaling
├── page.tsx              ← the landing page: header + LoadForm + stored projects
└── api/projects/route.ts ← receives the zip, streams the stages back
components/
├── app-header.tsx        ← server: name + ThemeControl
├── theme-control.tsx     ← client: system | light | dark
├── load-form.tsx         ← client: file input, Analyse, stage list
└── state-mark.tsx        ← the small circle that shows queued / running / complete / failed
lib/
├── theme.ts              ← the three themes and the cookie name
├── time.ts               ← "3h ago"
├── ingest/stages.ts      ← the list of stages, shared by server and browser
└── store/projects.ts     ← listProjects() reads data/<id>/meta.json
```

### Step-1.1 Shell and look

> ### <span style="color: #3b82f6">Q: what is `layout.tsx`, and why is the theme read there instead of in the browser?</span>
>
> `app/layout.tsx` is the **root layout**: Next.js wraps every page in it, and it stays mounted while you move between pages. It is the only place `<html>` and `<body>` are written.
>
> ```tsx
> export default async function RootLayout({ children }: LayoutProps<"/">) {
>   const theme = parseTheme((await cookies()).get(THEME_COOKIE)?.value);
>   return (
>     <html lang="en" data-theme={theme} className={`${geistSans.variable} ${geistMono.variable} h-full antialiased`}>
>       <body className="flex h-full flex-col">{children}</body>
>     </html>
>   );
> }
> ```
>
> **The theme is a cookie, not `localStorage`.** The server can read a cookie while it builds the HTML, so `data-theme="dark"` is already on `<html>` in the first byte. `localStorage` is only visible to the browser, which would mean either a flash of the wrong theme or a small script that runs before paint. `lib/theme.ts` keeps the three allowed values and `parseTheme` turns anything else (no cookie, a mangled one) into `"system"`.
>
> **`LayoutProps<"/">`** is a type Next.js 16 generates, so `children` is typed without writing it by hand.
>
> **`h-full` and `flex flex-col` on `<body>`** make the page exactly the window tall, a column the header and the content share. That is what lets the map page later fill the window without a scrollbar.
>
> **`globals.css`** does two things:
> - **Colour tokens.** `--canvas`, `--surface`, `--raised`, `--line`, `--fg`, `--fg-muted`, `--accent`, plus `--incoming` (green) and `--outgoing` (amber). Each is one `light-dark(light, dark)` value. `color-scheme` on `<html>` decides which side applies: `system` follows the OS, and `data-theme="light"` or `"dark"` forces it. So there is **one palette**, not two copies. `@theme inline` turns the tokens into Tailwind classes (`bg-surface`, `text-fg-muted`, `border-line`).
> - **`rem` scaling.** Every size is in `rem`, and `html { font-size: clamp(16px, 100vw / 90, 32px) }` is the single dial: up to a 1440px-wide window it is 16px, past that it grows in proportion, so a large monitor shows the laptop layout bigger instead of the same small layout floating in empty space.
>
> This matches the project rule: a dense developer tool, small type (`font-size: 0.8125rem` on `body`), colour only where it means something.

**What was implemented in this step:**
1. **`app/layout.tsx`:** fonts (Geist and Geist Mono, through `next/font/google`), `<html data-theme>` from the cookie, the full-height column body.
2. **`lib/theme.ts`:** `THEMES`, `THEME_COOKIE`, `parseTheme`. No React or Next imports, so server and browser can share it.
3. **`app/globals.css`:** the tokens, the Tailwind mapping, `rem` scaling.

### Step-1.2 Header and theme control

> ### <span style="color: #3b82f6">Q: why is the header a server component but the theme buttons a client component?</span>
>
> ```tsx
> // components/app-header.tsx   (server: no "use client")
> export async function AppHeader() {
>   const theme = parseTheme((await cookies()).get(THEME_COOKIE)?.value);
>   return (
>     <header className="flex h-9 shrink-0 items-center gap-3 border-b border-line bg-surface px-3">
>       <Link href="/" className="font-mono text-xs font-semibold tracking-tight hover:underline">cartograph</Link>
>       <div className="ml-auto"><ThemeControl initial={theme} /></div>
>     </header>
>   );
> }
> ```
>
> **The header itself has nothing to react to**: a link and a bar. As a server component it ships no JavaScript of its own. It reads the cookie only to hand `ThemeControl` the value that is already selected.
>
> **`ThemeControl` must be a client component** because it has to answer a click:
> ```tsx
> "use client";
> function applyTheme(theme: Theme) {
>   document.documentElement.dataset.theme = theme;                 // 1. the page changes now
>   document.cookie = `${THEME_COOKIE}=${theme}; path=/; max-age=31536000; samesite=lax`; // 2. remembered for next load
> }
> ```
> It also keeps `useState(initial)` so the selected button highlights. Setting `dataset.theme` is enough to restyle the whole page, because the tokens in `globals.css` hang off `data-theme`. No request is made.
>
> **It is a `radiogroup`** (`role="radio"`, `aria-checked`): three mutually exclusive choices, so a keyboard and a screen reader treat it that way. `shrink-0` on the header and `h-9` keep it a fixed 36px bar.
>
> **`shrink-0` and `ml-auto`:** in a flex row, `ml-auto` pushes the theme control to the far right; `shrink-0` stops the column layout from squashing the header when the content below is tall.

**What was implemented in this step:**
1. **`components/app-header.tsx`:** the bar, as a server component.
2. **`components/theme-control.tsx`:** the three buttons, as a client component.

### Step-1.3 The stored projects list

> ### <span style="color: #3b82f6">Q: how does the page get the list, and why does it call `connection()`?</span>
>
> `app/page.tsx` is an **async server component**, so it can simply `await` the data before returning JSX:
> ```tsx
> export default async function LandingPage() {
>   const { projects, now } = await load();
>   ...
> }
> async function load() {
>   await connection();
>   return { projects: await listProjects(), now: Date.now() };
> }
> ```
>
> **`listProjects()`** (in `lib/store/projects.ts`) reads the `data/` folder, and for each sub-folder whose name is a 16-character hex id it reads `meta.json`, checks the shape, and returns the records **newest first**. It reads only `meta.json`, not the large `result.json` or `sources.json`, so the list stays fast however big the projects are. If `data/` does not exist yet, that is an empty list, not an error.
>
> **`await connection()`** (from `next/server`) says "wait for a real request before running the next line". Without it Next.js may try to render the page once, ahead of time, and serve that same HTML to everyone: the list would be frozen at whatever `data/` held at build time. The docs describe it as the call for a component that needs different output per request. Here, the folder changes after every load.
>
> **`now` is taken once, on the server**, and passed to `ago(p.addedAt, now)` (`lib/time.ts`), which turns a timestamp into `45s ago`, `3h ago`, `2d ago`. Nothing ticks in the browser: the text is right at the moment the page is built, and the exact time is in the `title` of the `<time>` element. A relative time rather than a clock time also avoids the server and the browser disagreeing about a timezone.
>
> **The table** is a fixed layout (`table-fixed` and a `<colgroup>`): the name column takes what is left, the four number columns are fixed widths and right-aligned with `tabular-nums` so digits line up. Names are in monospace (`font-mono`). Each name is a `<Link href="/projects/<id>">`; clicking it opens the stored map and re-parses nothing.
>
> **"N skipped, N unresolved"** appears beside a name only when either is above zero. That is the project's rule in the UI: a skipped file is *said and counted*, not hidden. There is **no score** and nothing graded.
>
> **Empty state:** "None yet. A project appears here once it has been loaded."

**What was implemented in this step:**
1. **`lib/store/projects.ts`: `listProjects` and `readMeta`** (the rest of the store is used by later phases).
2. **`lib/time.ts`:** `ago`.
3. **`app/page.tsx`:** the header, the `LoadForm`, and the table, with the empty state.

### Step-1.4 The load form

> ### <span style="color: #3b82f6">Q: what is on screen in the form, and how does it show progress?</span>
>
> `components/load-form.tsx` is a client component with two pieces of state:
> ```tsx
> const [file, setFile] = useState<File | null>(null);   // the chosen zip
> const [run, setRun] = useState<Run>({ phase: "idle" });  // where the load has got to
>
> type Run =
>   | { phase: "idle" }
>   | { phase: "uploading"; percent: number }
>   | { phase: "running"; stage: Stage; message: string }
>   | { phase: "failed"; stage: Stage | "upload"; message: string };
> ```
> **`Run` is a union, not several booleans.** `uploading`, `running` and `failed` can never be true together, and each carries only the data that belongs to it (a percent while uploading, a stage and message while running). TypeScript then refuses code that reads `run.percent` when the run is not uploading.
>
> **The controls.** An `<input type="file" accept=".zip,application/zip">` and an **Analyse** button. The button is disabled until a file is chosen and while a load is `busy`, so a double click cannot start two uploads.
>
> **The stage list** appears as soon as `run.phase` is not `idle`: first **Upload**, then one row per entry in `STAGES`. Each row is `[mark] label detail` on a three-column grid, so the marks and labels line up.
>
> **`lib/ingest/stages.ts` is the one list of stages:**
> ```ts
> export const STAGES = [
>   { stage: "extract", label: "Unpack zip" },
>   { stage: "select",  label: "Select files" },
>   { stage: "parse",   label: "Parse imports" },
>   { stage: "store",   label: "Store project" },
> ];
> ```
> The server reports a stage by its short name, and the browser shows its label. Because the file has **no Node imports**, both sides can import it, and the list can't drift apart.
>
> **Each row's state** is worked out from the position of the current stage: rows before it are `complete`, the row itself is `running`, the rest are `queued`; if the run failed, that row is `failed` and shows the server's message. **`StateMark`** draws the state as a shape, not a colour: an empty ring (queued), a half-filled ring (running), a filled disc (complete), a crossed ring (failed). Green and amber already mean *direction* on the map, so status stays greyscale.
>
> A line under the input says what the tool accepts and that nothing leaves the machine: *"A zip of a TypeScript or JavaScript project. Dependencies and dot-folders are left out; nothing leaves this machine."*

**What was implemented in this step:**
1. **`lib/ingest/stages.ts`:** `Stage` and `STAGES`.
2. **`components/state-mark.tsx`:** the four marks.
3. **`components/load-form.tsx`:** the input, the button and the stage list, driven by `Run`. The sending (next step) is the `start` function.

### Step-1.5 Sending the zip and streaming the stages back

> ### <span style="color: #3b82f6">Q: how does a click become a stored project, and how does the page know which stage it is at?</span>
>
> **Browser side, `start()` in `load-form.tsx`:**
> ```tsx
> const request = new XMLHttpRequest();
> request.open("POST", `/api/projects?name=${encodeURIComponent(file.name)}`);
> request.upload.onprogress = (e) => setRun({ phase: "uploading", percent: ... });
> request.onprogress = consume;       // new bytes of the answer have arrived
> request.send(file);                 // the zip itself is the body
> ```
> - **`XMLHttpRequest`, not `fetch`**, because `fetch` cannot report how much of an upload has been sent. The percent in the **Upload** row needs `request.upload.onprogress`.
> - **The file is the body**, not a form field. The browser can send it straight from disk without reading it into memory first, which matters for a zip of hundreds of megabytes.
> - **The name** travels in the URL (`?name=`) and is only a label for the list.
>
> **Server side, `app/api/projects/route.ts`:**
> 1. **`cleanName`** strips `.zip` and any character that is not a letter, digit, `.`, `_`, space or `-`, and cuts it to 80 characters. The name is a label only and **never touches a path**.
> 2. **`saveBody`** copies the request body to a file in `data/` **chunk by chunk**, counting bytes. Over 4 GB, or nothing at all, throws and the partial file is removed.
> 3. A **`ReadableStream`** is returned as the response. Inside it, `ingestZip(zipPath, name, report)` runs the four stages, and the `report` callback calls `send(...)` each time a stage changes.
> 4. **`finally { rm(zipPath) }`**: whatever happens, the temporary zip is removed. Your original zip was never touched; this is the copy that arrived.
>
> **The answer is a stream of JSON lines** (`application/x-ndjson`), one event per line:
> ```
> {"type":"stage","stage":"extract","message":"…"}
> {"type":"stage","stage":"select","message":"…"}
> {"type":"stage","stage":"parse","message":"…"}
> {"type":"stage","stage":"store","message":"…"}
> {"type":"done","id":"3f9a…","alreadyStored":false}      ← or {"type":"error","stage":"parse","message":"…"}
> ```
> `IngestEvent` is exported from the route and imported (as a **type** only) by the form, so both sides agree on the shape.
>
> **Reading the stream in the browser.** `consume()` runs on every `onprogress`. It takes the new text since the last read, **only up to the last newline** (a half-arrived line is left for next time), splits it into lines, `JSON.parse`s each, and calls `setRun` for a `stage` event or remembers a `done`/`error` event. No polling, no timer: the stage on screen is the stage the server just sent.
>
> **The end.** On `done` the page calls `router.push("/projects/<id>")`, so you land on the map. `alreadyStored` is true when the same code is loaded twice; it opens the existing project. On `error`, the failed row shows the stage and the server's message. If the connection drops, or the server stops before a final line, the form says so in words instead of hanging.

**What was implemented in this step:**
1. **`app/api/projects/route.ts`:** `POST`, `saveBody`, `cleanName`, the streamed response, the `IngestEvent` type.
2. **`components/load-form.tsx`:** `start()` and `consume()`.
3. **`lib/ingest/run.ts`:** `ingestZip` and `IngestError` already existed; the route only calls them (their inside is the next steps).

### Step-1 flow: what happens when you use the landing page

![How the landing page works: opening / runs layout.tsx and page.tsx on the server, which read data/ and send back HTML; clicking Analyse posts the zip to the route handler, which streams stage lines back; the theme buttons write a cookie](SVG%20Step-1%20Landing%20page/landing-page-flow.svg)

The diagram has three parts: **A** is opening `/`, **B** is clicking **Analyse**, and **C** is the theme buttons.

**A · Opening `/`**

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **The browser asks for `/`.** Next.js finds `app/page.tsx` for that route and wraps it in `app/layout.tsx`.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **`layout.tsx` reads the theme cookie** and writes `data-theme` onto `<html>`, so the page is the right colour from the first byte.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **`page.tsx` waits for the request (`connection()`), then calls `listProjects()`.** That reads `data/<id>/meta.json` for each stored project and returns them newest first.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **The server sends finished HTML**: header, form, table. Only `LoadForm` and `ThemeControl` ship JavaScript to become interactive.

**B · Clicking Analyse**

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **`LoadForm` sends the zip** with an `XMLHttpRequest` to `/api/projects`. The Upload row shows a percent.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **`route.ts` saves it to disk and runs `ingestZip`.** Each time a stage starts, it writes one JSON line to the open response.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **`LoadForm` reads each line as it arrives** and moves the ring from queued to running to complete.

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **On `done`, the page navigates to `/projects/<id>`.** On `error`, the failed row shows the stage and the reason.

**C · The theme buttons**

<span style="color: #2563eb; font-size: 1.15em; font-weight: bold;">➤</span> **`ThemeControl` changes `data-theme` straight away and writes a cookie.** No request. Next time, `layout.tsx` reads that cookie.

> **The one idea to take away:**
> ```
> data/ ─► page.tsx (server) ─► HTML ─► you see the list
> zip  ─► LoadForm ─► route.ts ─► stage lines ─► LoadForm ─► /projects/<id>
> ```
> The server owns the data and the work; the browser only shows what the server tells it. Nothing is polled.

### Step-1 checked, not checked, and what changes next

**Checked from the terminal:** types, lint and the production build pass. A 483 MB zip loaded through the same endpoint produced one project folder in `data/` and nothing else (see *First Build*).

**Not checked: anything that needs a browser.** How the page looks in light and dark, the Upload percent moving, the stage rings ticking, the list updating after a load, and the link into the map are yours to run.

**What the path-box change will touch** (not done):

| File | Change |
|---|---|
| `components/load-form.tsx` | The file input and `XMLHttpRequest` upload go. A text field holds the path; **Go** posts `{ path }`. The Upload row goes (nothing is uploaded), and the line under the field becomes *"Paste a path, or drag a zip or folder from Explorer into the box."* |
| `app/api/projects/route.ts` | Takes `{ path }` instead of the body; checks it is a `.zip` or a folder; `saveBody`, the upload limit and `finally { rm(zipPath) }` for the incoming copy go. |
| `lib/ingest/stages.ts` | A folder has no *Unpack zip*, so the list the form shows depends on the kind of path. |
| `app/page.tsx`, `layout.tsx`, `app-header.tsx`, `theme-control.tsx`, the table | **Unchanged.** |